# Trabalho 2 - Recuperação da Informação

Grupo:

- Arthur Trottmann Ramos (14681052)
- Maicon Chaves Marques (14593530)

## Instalação de Dependências

In [19]:
%pip install nltk numpy pandas matplotlib sentence-transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.


## Pré-processamento

A representação lexical usa normalização para minúsculas, tokenização, remoção de stopwords em inglês e stemming Porter. O mesmo pré-processamento será aplicado aos documentos e às consultas.

In [20]:
import json
import math
import re
import string
from collections import Counter, defaultdict
from time import perf_counter
from pathlib import Path

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

try:
    STOPWORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    STOPWORDS = set(stopwords.words('english'))
stemmer = PorterStemmer()


def tokenize(text):
    """Normaliza para minúsculas e retorna tokens sem stemming."""
    return re.findall(r"\b\w+\b", str(text).lower())


def preprocess_tokens(tokens):
    """Remove stopwords e aplica stemming a tokens já normalizados."""
    return [stemmer.stem(token) for token in tokens if token not in STOPWORDS]


def preprocess(text):
    """Tokeniza, normaliza, remove stopwords e aplica stemming Porter."""
    return preprocess_tokens(tokenize(text))


## Leitura do NF-Corpus

Os arquivos JSONL são carregados como dicionários indexados pelo identificador do documento ou da consulta. Os três arquivos de qrels são lidos separadamente e também reunidos em um único DataFrame.

In [21]:
# O notebook deve ser executado a partir da pasta raiz do projeto.
DATA_DIR = Path('nfcorpus/nfcorpus')


def read_jsonl(path):
    """Lê um arquivo JSON Lines e retorna registros indexados pelo campo _id."""
    records = {}
    with path.open(encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            record_id = record.get('_id')
            if record_id is None:
                raise ValueError(f'Registro sem _id em {path}, linha {line_number}')
            records[record_id] = record
    return records


documents = read_jsonl(DATA_DIR / 'corpus.jsonl')
queries = read_jsonl(DATA_DIR / 'queries.jsonl')

qrels = {
    split: pd.read_csv(DATA_DIR / 'qrels' / f'{split}.tsv', sep='\t')
    for split in ('train', 'dev', 'test')
}
qrels_all = pd.concat(qrels.values(), ignore_index=True)

print(f'Documentos: {len(documents):,}')
print(f'Consultas: {len(queries):,}')
print(f'Relevâncias (train/dev/test): {len(qrels_all):,}')
print('Colunas dos qrels:', list(qrels_all.columns))


Documentos: 3,633
Consultas: 3,237
Relevâncias (train/dev/test): 134,294
Colunas dos qrels: ['query-id', 'corpus-id', 'score']


## Construção do Índice Invertido

Cada documento é representado pela concatenação de título e texto. O índice associa cada termo processado aos documentos em que ocorre e à frequência do termo em cada documento. Também são armazenados o comprimento processado de cada documento e o total de documentos, usados no cálculo de BM25.

In [15]:
index_build_start = perf_counter()
inverted_index = defaultdict(dict)
document_lengths = {}
raw_vocabulary = set()

for doc_id, document in documents.items():
    title = document.get('title', '')
    text = document.get('text', '')
    raw_tokens = tokenize(f'{title} {text}')
    raw_vocabulary.update(raw_tokens)
    tokens = preprocess_tokens(raw_tokens)
    term_frequencies = Counter(tokens)
    document_lengths[doc_id] = len(tokens)

    for term, frequency in term_frequencies.items():
        inverted_index[term][doc_id] = frequency

# Converte para dict comum para facilitar inspeção e serialização posteriores.
inverted_index = dict(inverted_index)
index_build_time_seconds = perf_counter() - index_build_start
num_documents = len(documents)
avg_document_length = (
    sum(document_lengths.values()) / num_documents if num_documents else 0
)
num_postings = sum(len(postings) for postings in inverted_index.values())

def document_frequency(term):
    """Retorna em quantos documentos o termo processado aparece."""
    return len(inverted_index.get(term, {}))

print(f'Termos no vocabulário: {len(inverted_index):,}')
print(f'Total de postings: {num_postings:,}')
print(f'Tempo de construção do índice: {index_build_time_seconds:.4f} s')
print(f'Tamanho médio dos documentos: {avg_document_length:.2f} tokens')
print('Exemplo de postings:', next(iter(inverted_index.items())))


Termos no vocabulário: 19,054
Total de postings: 360,200
Tempo de construção do índice: 3.2794 s
Tamanho médio dos documentos: 165.82 tokens
Exemplo de postings: ('statin', {'MED-10': 12, 'MED-14': 6, 'MED-870': 1, 'MED-1193': 11, 'MED-1207': 2, 'MED-1258': 2, 'MED-1283': 1, 'MED-1429': 1, 'MED-1486': 3, 'MED-1554': 1, 'MED-1639': 1, 'MED-1874': 1, 'MED-1887': 2, 'MED-2428': 4, 'MED-2429': 8, 'MED-2431': 11, 'MED-2432': 1, 'MED-2525': 1, 'MED-3204': 1, 'MED-3430': 1, 'MED-4559': 9, 'MED-4562': 5, 'MED-4825': 1, 'MED-4827': 1, 'MED-4828': 8, 'MED-4829': 10, 'MED-4830': 8})


## Busca Lexical com BM25

A busca calcula pontuações apenas para documentos encontrados nas posting lists dos termos da consulta. O tempo médio pode ser medido sobre as consultas julgadas de uma partição, sem usar o resultado para ajustar parâmetros no conjunto de teste.

In [22]:
BM25_K1 = 1.2
BM25_B = 0.75


def search_bm25(query, top_k=10, k1=BM25_K1, b=BM25_B):
    """Retorna os top_k documentos ordenados pela pontuação BM25."""
    if k1 <= 0 or not 0 <= b <= 1:
        raise ValueError('BM25 requer k1 > 0 e 0 <= b <= 1')
    if top_k <= 0 or num_documents == 0 or avg_document_length == 0:
        return []

    query_terms = set(preprocess(query))
    scores = defaultdict(float)

    for term in query_terms:
        postings = inverted_index.get(term)
        if not postings:
            continue

        df = len(postings)
        # IDF BM25 com suavização, que mantém valores positivos.
        idf = math.log1p((num_documents - df + 0.5) / (df + 0.5))

        for doc_id, term_frequency in postings.items():
            doc_length = document_lengths[doc_id]
            length_norm = k1 * (1 - b + b * doc_length / avg_document_length)
            tf_weight = (term_frequency * (k1 + 1)) / (term_frequency + length_norm)
            scores[doc_id] += idf * tf_weight

    ranked = sorted(scores.items(), key=lambda item: (-item[1], item[0]))[:top_k]
    return [
        {'doc_id': doc_id, 'score': float(score)}
        for doc_id, score in ranked
    ]


def measure_bm25_query_time(split='dev', top_k=100, repetitions=1):
    """Mede a média por consulta para IDs julgados na partição indicada."""
    if split not in qrels:
        raise ValueError(f'Partição desconhecida: {split}. Use train, dev ou test.')
    if repetitions <= 0:
        raise ValueError('repetitions deve ser maior que zero')

    query_ids = sorted(set(qrels[split]['query-id']) & set(queries))
    if not query_ids:
        raise ValueError(f'Nenhuma consulta de {split} foi encontrada em queries.jsonl')

    start = perf_counter()
    for _ in range(repetitions):
        for query_id in query_ids:
            search_bm25(queries[query_id].get('text', ''), top_k=top_k)
    elapsed = perf_counter() - start
    total_queries = len(query_ids) * repetitions
    return {
        'split': split,
        'num_queries': len(query_ids),
        'repetitions': repetitions,
        'average_query_time_seconds': elapsed / total_queries,
    }


# Exemplo de uso após criar o índice:
# search_bm25(queries['PLAIN-3']['text'], top_k=10)
# Depois de fixar os parâmetros com dev, pode-se medir no conjunto final:
# measure_bm25_query_time(split='test', top_k=100)


## Tolerância a erros: k-gramas, Jaccard e distância de edição

O vocabulário de correção é formado pelos tokens do corpus antes de stemming, conforme o item 4 do PDF. Termos conhecidos são preservados; para termos fora do vocabulário, k-gramas geram candidatos, Jaccard filtra candidatos e distância de Levenshtein seleciona a alternativa mais próxima.

In [ ]:
KGRAM_SIZE = 3
JACCARD_THRESHOLD = 0.30
MAX_EDIT_DISTANCE = 2
TYPO_OPERATIONS = ('insertion', 'deletion', 'substitution', 'transposition')
TYPO_LABELS = {
    'insertion': 'inserção',
    'deletion': 'remoção',
    'substitution': 'substituição',
    'transposition': 'troca de ordem',
}


def character_kgrams(term, k=KGRAM_SIZE):
    """Retorna k-gramas de caracteres com marcadores de início e fim."""
    if k <= 0:
        raise ValueError('k deve ser maior que zero')
    padded = '$' * (k - 1) + term + '$' * (k - 1)
    if len(padded) < k:
        return {padded}
    return {padded[i:i + k] for i in range(len(padded) - k + 1)}


def jaccard_coefficient(left, right):
    """Calcula J(A,B) = |A ∩ B| / |A ∪ B|."""
    union = left | right
    return len(left & right) / len(union) if union else 1.0


def edit_distance(left, right):
    """Distância de Levenshtein usando inserção, remoção e substituição."""
    if len(left) < len(right):
        left, right = right, left
    previous = list(range(len(right) + 1))
    for i, left_char in enumerate(left, start=1):
        current = [i]
        for j, right_char in enumerate(right, start=1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j - 1] + (left_char != right_char),
            ))
        previous = current
    return previous[-1]


# Índice auxiliar de k-gramas construído a partir do vocabulário não stemmizado.
kgram_index_build_start = perf_counter()
kgram_index = defaultdict(set)
for vocabulary_term in raw_vocabulary:
    for gram in character_kgrams(vocabulary_term):
        kgram_index[gram].add(vocabulary_term)
kgram_index = dict(kgram_index)
kgram_index_build_time_seconds = perf_counter() - kgram_index_build_start
print(f'Termos no vocabulário de correção (antes do stemming): {len(raw_vocabulary):,}')
print(f'Tempo de construção do índice de k-gramas: {kgram_index_build_time_seconds:.4f} s')


def best_spelling_candidate(term, jaccard_threshold=JACCARD_THRESHOLD,
                            max_distance=MAX_EDIT_DISTANCE):
    """Encontra correção para termo OOV: k-gramas, Jaccard e Levenshtein."""
    if term in raw_vocabulary:
        return None

    query_grams = character_kgrams(term)
    candidates = set()
    for gram in query_grams:
        candidates.update(kgram_index.get(gram, set()))

    filtered = []
    for candidate in candidates:
        similarity = jaccard_coefficient(query_grams, character_kgrams(candidate))
        if similarity < jaccard_threshold:
            continue
        distance = edit_distance(term, candidate)
        if distance <= max_distance:
            filtered.append((distance, -similarity, candidate, similarity))

    if not filtered:
        return None
    distance, _, candidate, similarity = min(filtered)
    return {'term': candidate, 'distance': distance, 'jaccard': similarity}


def correct_query(query, jaccard_threshold=JACCARD_THRESHOLD,
                  max_distance=MAX_EDIT_DISTANCE):
    """Corrige apenas tokens desconhecidos e retorna texto e detalhes."""
    corrections = []

    def replace_unknown(match):
        observed = match.group(0).lower()
        choice = best_spelling_candidate(
            observed,
            jaccard_threshold=jaccard_threshold,
            max_distance=max_distance,
        )
        if choice is None:
            return match.group(0)
        corrections.append({
            'observed': observed,
            'replacement': choice['term'],
            'edit_distance': choice['distance'],
            'jaccard': choice['jaccard'],
        })
        return choice['term']

    corrected = re.sub(r'\b\w+\b', replace_unknown, str(query))
    return corrected, corrections


def _typo_candidates(term, operation):
    """Gera corrupções de um único tipo, em ordem determinística."""
    alphabet = string.ascii_lowercase
    if operation == 'insertion':
        return [term[:i] + char + term[i:]
                for i in range(len(term) + 1) for char in alphabet]
    if operation == 'deletion':
        return [term[:i] + term[i + 1:]
                for i in range(len(term)) if len(term) > 2]
    if operation == 'substitution':
        return [term[:i] + char + term[i + 1:]
                for i in range(len(term)) for char in alphabet if char != term[i]]
    if operation == 'transposition':
        return [term[:i] + term[i + 1] + term[i] + term[i + 2:]
                for i in range(len(term) - 1) if term[i] != term[i + 1]]
    raise ValueError(f'Tipo de erro desconhecido: {operation}')


def _replace_first_token(query, original, replacement):
    pattern = rf'\b{re.escape(original)}\b'
    return re.sub(pattern, replacement, query, count=1, flags=re.IGNORECASE)


def make_oov_typo(query, operation, jaccard_threshold=JACCARD_THRESHOLD,
                  max_distance=MAX_EDIT_DISTANCE):
    """Introduz um erro OOV que o corretor consegue mapear ao termo original."""
    query_terms = sorted(set(tokenize(query)), key=lambda term: (-len(term), term))
    for original in query_terms:
        if original not in raw_vocabulary:
            continue
        for typo in _typo_candidates(original, operation):
            if typo in raw_vocabulary:
                continue
            choice = best_spelling_candidate(
                typo,
                jaccard_threshold=jaccard_threshold,
                max_distance=max_distance,
            )
            if choice is None or choice['term'] != original:
                continue
            typo_query = _replace_first_token(query, original, typo)
            if typo_query != query:
                return {
                    'query': typo_query,
                    'original_term': original,
                    'typo_term': typo,
                    'corrected_term': choice['term'],
                }
    return None


## Experimento de tolerância a erros (item 4 do PDF)

O experimento seleciona 10 consultas distintas de `test` com NDCG@10 original maior que zero. Cada uma recebe uma corrupção OOV de um dos quatro tipos de edição. São comparadas as versões original, com erro e corrigida. O mesmo mecanismo também aceita `dev` para ajustes de limiar; os demais experimentos devem continuar usando as consultas originais.

In [ ]:
def _qrels_by_query(qrels_frame):
    judgments = defaultdict(dict)
    for _, row in qrels_frame.iterrows():
        qid = row['query-id']
        doc_id = row['corpus-id']
        judgments[qid][doc_id] = max(
            float(row['score']), judgments[qid].get(doc_id, float('-inf'))
        )
    return dict(judgments)


def ndcg_at_k(ranked_doc_ids, relevant_grades, k=10):
    """NDCG com ganho linear e IDCG calculado sobre todos os qrels da consulta."""
    if k <= 0:
        return 0.0
    dcg = sum(
        relevant_grades.get(doc_id, 0.0) / math.log2(rank + 2)
        for rank, doc_id in enumerate(ranked_doc_ids[:k])
    )
    ideal_grades = sorted(relevant_grades.values(), reverse=True)[:k]
    idcg = sum(grade / math.log2(rank + 2)
               for rank, grade in enumerate(ideal_grades))
    return dcg / idcg if idcg > 0 else 0.0


def _bm25_ndcg(query_text, query_id, judgments, k=10):
    results = search_bm25(query_text, top_k=k)
    return ndcg_at_k(
        [result['doc_id'] for result in results],
        judgments.get(query_id, {}),
        k=k,
    )


def run_typo_experiment(n_queries=10, split='test', k=10,
                        jaccard_threshold=JACCARD_THRESHOLD,
                        max_distance=MAX_EDIT_DISTANCE):
    """Roda experimento original/erro/corrigida no BM25 em uma partição."""
    if split not in qrels:
        raise ValueError(f'Partição desconhecida: {split}. Use train, dev ou test.')
    if split == 'test' and n_queries != 10:
        raise ValueError('O protocolo do PDF exige exatamente 10 consultas de test.')
    if n_queries <= 0:
        raise ValueError('n_queries deve ser maior que zero')

    judgments = _qrels_by_query(qrels[split])
    query_ids = sorted(set(judgments) & set(queries))
    records = []
    next_operation = 0

    for query_id in query_ids:
        if len(records) == n_queries:
            break
        original_query = queries[query_id].get('text', '')
        original_ndcg = _bm25_ndcg(original_query, query_id, judgments, k=k)
        if original_ndcg <= 0:
            continue

        operation = TYPO_OPERATIONS[next_operation % len(TYPO_OPERATIONS)]
        typo = make_oov_typo(
            original_query,
            operation,
            jaccard_threshold=jaccard_threshold,
            max_distance=max_distance,
        )
        if typo is None:
            continue

        error_ndcg = _bm25_ndcg(typo['query'], query_id, judgments, k=k)
        corrected_query, corrections = correct_query(
            typo['query'],
            jaccard_threshold=jaccard_threshold,
            max_distance=max_distance,
        )
        corrected_ndcg = _bm25_ndcg(corrected_query, query_id, judgments, k=k)
        denominator = original_ndcg - error_ndcg
        if denominator > 0:
            recovered_fraction = (corrected_ndcg - error_ndcg) / denominator
            recovery_status = 'calculada'
        else:
            recovered_fraction = None
            recovery_status = 'erro não reduziu o desempenho'

        records.append({
            'query_id': query_id,
            'query_original': original_query,
            'query_erro': typo['query'],
            'query_corrigida': corrected_query,
            'tipo_erro': TYPO_LABELS[operation],
            'token_original': typo['original_term'],
            'token_com_erro': typo['typo_term'],
            'token_corrigido': typo['corrected_term'],
            'correcoes_aplicadas': json.dumps(corrections, ensure_ascii=False),
            'NDCG10_original': original_ndcg,
            'NDCG10_erro': error_ndcg,
            'NDCG10_corrigida': corrected_ndcg,
            'fracao_desempenho_recuperada': recovered_fraction,
            'status_recuperacao': recovery_status,
        })
        next_operation += 1

    if len(records) != n_queries:
        raise RuntimeError(
            f'Foram encontradas apenas {len(records)} consultas elegíveis; '
            f'eram necessárias {n_queries}. Verifique os limiares no conjunto dev.'
        )
    return pd.DataFrame(records)


# Experimento final: 10 consultas de test, após fixar parâmetros usando dev.
typo_results = run_typo_experiment(
    n_queries=10,
    split='test',
    k=10,
    jaccard_threshold=JACCARD_THRESHOLD,
    max_distance=MAX_EDIT_DISTANCE,
)
typo_results.to_csv('Experiments/typo_experiment.csv', index=False)
display(typo_results)
